<a href="https://colab.research.google.com/github/antoinewrd1/End-to-End-Generative-AI-Medical-Chatbot/blob/main/MedAssistAI_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MedAssist AI: Retrieval-Augmented Clinical Symptom Intelligence Platform

Tech Stack:

FastAPI (backend)

Streamlit (frontend)

OpenAI (LLM)

FAISS (RAG-ready layer placeholder)

CORS enabled

Docker-ready structure


# Architecture

Folder Structure

medassist-ai/
│
├── backend/
│   ├── main.py
│   │
│   ├── api/
│   │   └── routes.py
│   │
│   ├── services/
│   │   ├── llm_service.py
│   │   ├── rag_service.py
│   │   ├── fhir_service.py
│   │
│   ├── database/
│   │   ├── db.py
│   │   └── models.py
│   │
│   ├── config.py
│   └── prompts.py
│
├── frontend/
│   └── app.py
│
├── data/                # put CDC/WHO docs here
├── embeddings/          # FAISS index
│
├── Dockerfile
├── docker-compose.yml
├── requirements.txt
└── .env

# Requirements

fastapi

uvicorn

openai

pydantic

python-dotenv

streamlit

requests

sqlalchemy

psycopg2-binary

langchain

faiss-cpu

tiktoken

fhir.resources

# ENV CONFIG

In [ ]:
OPENAI_API_KEY=your_api_key_here
DATABASE_URL=postgresql://postgres:password@db:5432/medassist

# RAG (Retrieval-Augmented Generation)

services/rag_service.py

In [ ]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

def load_vector_db():
  embeddings = OpenAIEmbeddings()
  db = FAISS.load_local("embeddings", embeddings)
  return db

def retrieve_context(query: str):
  db = load_vector_db()
  docs = db.similarity_search(query, k=3)
  return "\n".join([doc.page_content for doc in docs])

# LLM WITH RAG

services/llm_service.py

In [ ]:
from openai import OpenAI
from config import OPENAI_API_KEY
from prompts import SYSTEM_PROMPT, build_prompt
from services.rag_service import retrieve_context

client = OpenAI(api_key=OPENAI_API_KEY)

def generate_response(symptoms: str):

  context = retrieve_context(symptoms)

  augmented_prompt = f"""
Clinical Context:
{context}

{build_prompt(symptoms)}
"""

    response = client.chat.completions.create(
      model="gpt-4o-mini",
      messages=[
          {"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": augmented_prompt}
      ],
      temperature=0.2
    )


    return response.choices[0].message.content

# FHIR OUTPUT

services/fhir_service.py

In [ ]:
from fhir.resources.observation import Observation

def create_fhir_observation(symptoms: str, result:str):

  obs = Observation.construct()

  obs.status = "final"
  obs.code = {
      "text": "Symptom Analysis"
  }

  obs.valueString = result

  return obs.json()

# DATABASE LOGGING

database/db.py

In [ ]:
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
import os

DATABASE_URL = os.getenv("DATABASE_URL")

engine = create_engine(DATABASE_URL)
SessionLocal = sessionmaker(bind=engine)

database/models.py

In [ ]:
from sqlalchemy import Column, Integer, String, Text
from sqlalchemy.orm import declarative_base

Base = declarative_base()

class QueryLog(Base):
  __tablename__ = "query_logs"

  id = Column(Integer, primary_key=True, index=True)
  symptoms = Column(Text)
  response = Column(Text)

# API ROUTES

api/routes.py

In [ ]:
from fastapi import APIRouter
from pydantic import BaseModel
from services.llm_service import generate_response
from services.fhir_service import create_fhir_observation
from database.db import SessionLocal
from database.models import QueryLog

router = APIRouter()

class SymptomRequest(BaseModel):
  symptoms: str

@router.post("/analyze")
def analyze(request: SymptomRequest):

  result = generate_response(request.symptoms)
  fhir = create_fhir_observation(request.symptoms, result)

  #Save to DB
  db = SessionLocal()
  log = QueryLog(symptoms=request.symptoms, response=result)
  db.add(log)
  db.commit()

  return {
      "response": result,
      "fhir": fhir
  }

# MAIN APP (WITH CORS)

backend/main.py

In [ ]:
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from api.routes import router

app = FastAPI(title="AI Medical Chatbot")

#CORS Configuration
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

app.include_router(router)

@app.get("/")
def root():
  return {"status": "running"}

# frontend/app.py (Streamlit Frontend)

In [ ]:
import streamlit as st
import requests
import json

API_URL = "http://localhost:8000/analyze"

st.set_page_config(
    page_title="MedAssist AI",
    layout="wide"
)

st.title("🩺 MedAssist AI")
st.caption("AI-powered clinical symptom exploration (educational use only)")

# Input
symptoms = st.text_area("Enter symptoms", height=150)

# Button
if st.button("Analyze"):

    if symptoms.strip():

        with st.spinner("Running clinical analysis..."):

            try:
                # ✅ FIX: correct JSON format
                res = requests.post(
                    API_URL,
                    json={"symptoms": symptoms}
                )

                # ✅ FIX: parse response correctly
                data = res.json()

                # -------------------------
                # DISPLAY RESULTS
                # -------------------------

                st.subheader("🧠 Clinical Output")
                st.write(data.get("response", "No response returned"))

                st.subheader("⚠️ Risk Level")
                st.write(data.get("risk", "Not provided"))

                st.subheader("🏥 Possible Codes")
                st.json(data.get("codes", []))

                st.subheader("FHIR Output")

                # ✅ FIX: safe JSON handling
                fhir_data = data.get("fhir", None)

                if fhir_data:
                    try:
                        st.json(json.loads(fhir_data))
                    except:
                        st.text(fhir_data)
                else:
                    st.warning("No FHIR data returned")

            except Exception as e:
                st.error(f"Request failed: {str(e)}")

    else:
        st.warning("Please enter symptoms before analyzing.")

# Dockerize

Dockerfile

In [ ]:
FROM python:3.11

WORKDIR / app

COPY . .

RUN pip install -r requirements.txt

CMD['uvicorn', "backend.main.app"], "--host", "0.0.0.0", "--port", "8000"

# docker-compose.yml

In [ ]:
version: "3.9"

services:
  backend:
  build:
  ports:
    - "8000:8000"
  env_file:
    - .env
  depends_on:
  - db

db:
  image: postgres
  restart: always
  environment:
    POSTGRES_DB: medassist
    POSTGRES_USER: postgres
    POSTGRES_PASSWORD: password
  ports:
    - "5432:5432"

# Run the App

Start Backend:

In [ ]:
Bash

docker-compose up --build

Start Frontend:

In [ ]:
Bash

cd frontend
streamlit run app.py

# Test Flow

Open Streamlit UI

Enter symptoms:

Click Analyze

Streamlit calls FastAPI

FastAPI calls OpenAI

Response returned and displayed
